# The Bergomi-Guyon coefficient recursion

A worked example through order three of the recursion in Bourgey and Gatheral (2026).

At fixed maturity $T$, using the paper's zero-carry convention,

$$k=\log(K/S_0),\qquad \Sigma(k)=T\,\sigma_{\mathrm{BS}}^2(k,T)
=M+\sum_{\ell\geq1}\epsilon^\ell a_\ell(k).$$

Here $M>0$ is the variance contract and $\epsilon$ records formal order; set $\epsilon=1$ after truncation. We use

$$\zeta=\frac12+\frac{k}{M},\qquad \theta=\frac1M,\qquad \kappa=\zeta-\frac12.$$

The recursion treats $\theta$ as a formal variable, allowing polynomial boundary values at $\theta=0$.

## Diamond trees and their order

The model inputs are diamond trees, built from $X=\log S$ and the variance contract $M_t(T)=\mathbb E_t[\langle X\rangle_T-\langle X\rangle_t]$, using

$$(A\diamond B)_t(T)=\mathbb E_t[\langle A,B\rangle_T]-\langle A,B\rangle_t.$$

The diamond product is commutative but not associative: bracketing specifies the tree shape. For a tree $\tau$ with $m$ leaves $M$ and $x$ leaves $X$,

$$\operatorname{ord}(\tau)=2m+x-2,\qquad
r_\tau(a)=\left[\frac12a(a-1)\right]^{m-1}a^x.$$

Start with $M$ of weight one. Joining $X$ to a tree preserves its weight; joining two generated trees multiplies their weights, with an extra factor $1/2$ when the children are identical. The boundary contribution is $w_\tau r_\tau(\zeta)\tau$; a chosen model supplies the numerical tree values.

In [1]:
import sympy as sp
from IPython.display import display

import bergomi_guyon as bg
from bergomi_guyon.render import tree_to_text

MAX_ORDER = 2
result = bg.generate_coefficients(MAX_ORDER)
forests = result.trees
boundaries = result.boundaries
coefficients = result.coefficients
sources = result.sources
zeta, theta, k, M = sp.symbols("zeta theta k M")

Each coefficient dictionary maps a tuple of trees to an exact polynomial in $(\zeta,\theta)$: `(tree,)` denotes a single tree and `(tree, tree)` its square. The collapsed helper cell converts these objects into readable formulas.

In [2]:
# Give each formal tree a short display name T_{order,number}.
tree_symbol = {
    tree: sp.Symbol(f"T_{{{order},{number}}}")
    for order, trees in enumerate(forests)
    for number, tree in enumerate(sorted(trees), start=1)
}


def polynomial_as_sympy(poly):
    """Convert the generator's exact sparse polynomial to SymPy."""
    return sp.Add(
        *(
            sp.Rational(value.numerator, value.denominator) * zeta**z * theta**t
            for (z, t), value in poly.terms.items()
        )
    )


def forest_as_sympy(forest_poly):
    """Convert a polynomial whose coefficients are tree monomials."""
    return sp.expand(
        sp.Add(
            *(
                polynomial_as_sympy(poly)
                * sp.prod(tree_symbol[tree] for tree in monomial)
                for monomial, poly in forest_poly.items()
            )
        )
    )

## The recursion recipe

The BG operator is the finite backward heat transform

$$\mathcal B[q]=\sum_{j\geq0}\frac{(-\theta/2)^j}{j!}q^{(2j)}(\zeta),
\qquad \mathcal B[a]=\zeta,\quad \mathcal B[a^2]=\zeta^2-\theta.$$

It satisfies $D\mathcal B[q]=0$ with $D=\partial_\theta+\tfrac12\partial_\zeta^2$. Single-tree prefactors are $w_\tau\mathcal B[r_\tau]$.

For products of trees, the paper's recursion theorem turns option-price matching into a nonlinear heat equation. Its universal input is the formal cumulant series

$$K(y)=\log\mathcal B[e^{y\lambda_a}]
=\frac{y\kappa^2}{2(1+\theta y)}-\frac12\log(1+\theta y)-\frac y8,
\qquad \lambda_a=\frac12a(a-1).$$

Writing $\widetilde{\Sigma}=\Sigma-M=\sum_{\ell\geq1}\epsilon^\ell a_\ell$, the theorem gives

$$D\widetilde{\Sigma}=-\partial_\zeta\widetilde{\Sigma}
\left[\frac12\partial_yK\,\partial_\zeta\widetilde{\Sigma}+\partial_\zeta K\right]_{y=\widetilde{\Sigma}}.$$

Take the partial derivatives before substituting $y=\widetilde{\Sigma}$. Here $S_\ell$ denotes the coefficient of $\epsilon^\ell$ on the right-hand side, which uses only lower orders. Solve

$$Da_\ell=S_\ell,\qquad
a_\ell(\zeta,0)=b_\ell(\zeta)
=\sum_{\tau\in\mathcal T_\ell}w_\tau r_\tau(\zeta)\tau.$$

The boundary contains only single trees; products have zero boundary value. We now work through the calculation at order two.

## Step 1: trees and boundary data

Generate the tree shapes and weights through order two, including the base leaf $M$.

In [3]:
for order, trees in enumerate(forests):
    for tree, weight in sorted(trees.items()):
        m_leaves, x_leaves = tree.leaves
        # Check that the leaf counts give the expected tree order.
        assert order == 2 * m_leaves + x_leaves - 2
        print(
            f"{tree_symbol[tree]} = {tree_to_text(tree):25s} "
            f"leaves=(M:{m_leaves}, X:{x_leaves}), weight={weight}"
        )

T_{0,1} = M                         leaves=(M:1, X:0), weight=1
T_{1,1} = (X diamond M)             leaves=(M:1, X:1), weight=1
T_{2,1} = (M diamond M)             leaves=(M:2, X:0), weight=1/2
T_{2,2} = (X diamond (X diamond M)) leaves=(M:1, X:2), weight=1


The order-one tree $T_{1,1}=X\diamond M$ gives $b_1=T_{1,1}\zeta$. At order two, $T_{2,1}=M\diamond M$ has weight $1/2$, while $T_{2,2}=X\diamond(X\diamond M)$ has weight one, so

$$b_2=\frac14T_{2,1}(\zeta^2-\zeta)+T_{2,2}\zeta^2.$$

Applying $\mathcal B$ gives the single-tree prefactors

$$T_{1,1}:\ \zeta,\qquad
T_{2,1}:\ \frac14(\zeta^2-\zeta-\theta),\qquad
T_{2,2}:\ \zeta^2-\theta.$$

## Step 2: the nonlinear second-order term

Write $\widetilde{\Sigma}=\epsilon a_1+\epsilon^2a_2+\cdots$ and $K(y)=K_1y+O(y^2)$, where

$$K_1=\frac12(\zeta^2-\zeta-\theta),\qquad
\partial_\zeta K_1=\zeta-\frac12.$$

Since $\widetilde{\Sigma}=O(\epsilon)$, the order-two source uses only $a_1=T_{1,1}\zeta$:

$$S_2=-\frac12K_1(\partial_\zeta a_1)^2
-(\partial_\zeta K_1)a_1\partial_\zeta a_1
=T_{1,1}^2\left(-\frac54\zeta^2+\frac34\zeta+\frac14\theta\right).$$

Thus the only nonlinear product is $T_{1,1}^2$. Write its prefactor as $c=c_{T_{1,1}^2}$; it solves

$$Dc=-\frac54\zeta^2+\frac34\zeta+\frac14\theta,\qquad c(\zeta,0)=0.$$

To compute the Taylor coefficients in $\theta$, hold $\zeta$ fixed and rearrange the PDE:

$$\partial_\theta c
=-\frac54\zeta^2+\frac34\zeta+\frac14\theta
-\frac12\partial_\zeta^2c.$$

Because $c(\zeta,0)=0$ for every $\zeta$, its $\zeta$-derivatives also vanish at $\theta=0$. Evaluating the PDE there gives the first derivative:

$$\partial_\theta c(\zeta,0)=-\frac54\zeta^2+\frac34\zeta,$$

For the second derivative, first differentiate the PDE in $\theta$. The polynomial derivatives commute, so

$$\partial_\theta^2c=\frac14-\frac12\partial_\zeta^2(\partial_\theta c).$$

Now set $\theta=0$ and insert the first derivative just computed:

$$\partial_\theta^2c(\zeta,0)
=\frac14-\frac12\partial_\zeta^2
\left(-\frac54\zeta^2+\frac34\zeta\right)=\frac32.$$

Differentiating once more gives $\partial_\theta^3c(\zeta,0)=-\tfrac12\partial_\zeta^2(3/2)=0$; the same recurrence makes every higher derivative zero. Thus the Taylor series terminates:

$$c(\zeta,\theta)=c(\zeta,0)+\theta\,\partial_\theta c(\zeta,0)
+\frac{\theta^2}{2!}\partial_\theta^2c(\zeta,0).$$

Substituting these values gives

$$c=\theta\left(-\frac54\zeta^2+\frac34\zeta\right)
+\frac{\theta^2}{2!}\frac32
=-\frac\theta4(5\zeta^2-3\zeta-3\theta).$$

Check this solution against the generator and its PDE:

In [21]:
T11 = next(iter(forests[1]))  # T_{1,1} = X diamond M.

product = (T11, T11)  # T_{1,1}^2.

# S_2 / T_{1,1}^2: the right-hand side of the equation for c.
source_2 = polynomial_as_sympy(sources[2][product])

# c(zeta, theta) = c_{T_{1,1}^2}: the prefactor of T_{1,1}^2 in a_2.
product_coefficient = polynomial_as_sympy(coefficients[2][product])

# The same c(zeta, theta), calculated from the Taylor expansion above.
taylor_coefficient = theta * (-5 * zeta**2 + 3 * zeta) / 4 + 3 * theta**2 / 4

# Check that the two expressions for c agree exactly.
assert sp.expand(product_coefficient - taylor_coefficient) == 0

# Check D c = S_2 / T_{1,1}^2.
Dc = sp.diff(product_coefficient, theta) + sp.diff(product_coefficient, zeta, 2) / 2
assert sp.expand(Dc - source_2) == 0

# Check the boundary condition c(zeta, 0) = 0.
assert product_coefficient.subs(theta, 0) == 0

print("Taylor solution, PDE, and boundary checks passed.")

Taylor solution, PDE, and boundary checks passed.


## Step 3: assemble the smile coefficients

Combine the single-tree prefactors with the product correction to obtain $a_1$ and $a_2$:

In [22]:
for order in range(1, MAX_ORDER + 1):
    display(sp.Eq(sp.Symbol(f"a_{order}"), forest_as_sympy(coefficients[order])))

Eq(a_1, T_{1,1}*zeta)

Eq(a_2, 3*T_{1,1}**2*theta**2/4 - 5*T_{1,1}**2*theta*zeta**2/4 + 3*T_{1,1}**2*theta*zeta/4 - T_{2,1}*theta/4 + T_{2,1}*zeta**2/4 - T_{2,1}*zeta/4 - T_{2,2}*theta + T_{2,2}*zeta**2)

## Extension: order three

Three new tree shapes appear:

$$T_{3,1}=M\diamond(X\diamond M),\qquad
T_{3,2}=X\diamond(M\diamond M),\qquad
T_{3,3}=X\diamond(X\diamond(X\diamond M)).$$

The first two have the same leaf counts $(m,x)=(2,1)$, but different shapes and weights $1$ and $1/2$. Their single-tree prefactors therefore differ by a factor of two. The third has weight one.

In [7]:
third_order = bg.generate_coefficients(3)
tree_symbol.update(
    {
        tree: sp.Symbol(f"T_{{3,{number}}}")
        for number, tree in enumerate(sorted(third_order.trees[3]), start=1)
    }
)

The six monomials in $a_3$ split into three single trees, two mixed products, and the cube $T_{1,1}^3$. Display these groups as $a_3^{[1]}$, $a_3^{[2]}$, and $a_3^{[3]}$: $a_3^{[j]}$ is the coefficient of $\epsilon^3$ in the paper's $\widetilde{\Sigma}_j$, which collects products of $j$ trees.

In [8]:
for j in (1, 2, 3):
    group = sp.Add(
        *(
            sp.factor(polynomial_as_sympy(poly))
            * sp.prod(tree_symbol[tree] for tree in monomial)
            for monomial, poly in sorted(third_order.coefficients[3].items())
            if len(monomial) == j
        )
    )
    display(sp.Eq(sp.Symbol(f"a_3^{{[{j}]}}"), group))

bg.verify(third_order)
print("All exact checks passed through order three.")

Eq(a_3^{[1]}, -T_{3,1}*(3*theta*zeta - theta - zeta**3 + zeta**2)/2 - T_{3,2}*(3*theta*zeta - theta - zeta**3 + zeta**2)/4 - T_{3,3}*zeta*(3*theta - zeta**2))

Eq(a_3^{[2]}, T_{1,1}*T_{2,1}*theta*(14*theta*zeta - 6*theta - 8*zeta**3 + 10*zeta**2 - 3*zeta)/8 + T_{1,1}*T_{2,2}*theta*(14*theta*zeta - 3*theta - 8*zeta**3 + 5*zeta**2)/2)

Eq(a_3^{[3]}, -T_{1,1}**3*theta**2*(32*theta*zeta - 10*theta - 26*zeta**3 + 24*zeta**2 - 5*zeta)/8)

All exact checks passed through order three.


The verifier checks the PDE, boundary data, tree weights and completeness, degree structure, and original matching identity using exact rational arithmetic.

## References

Bourgey, F., & Gatheral, J. (2026). *Demystifying the Bergomi–Guyon expansion* [Manuscript].